# Dataset loading 

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tokenization

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 1000/1000 [00:00<00:00, 7146.59 examples/s]


# Adapter model

In [3]:
import torch
import torch.nn as nn
from transformers import BertModel

class Adapter(nn.Module):
    def __init__(self, hidden=768, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(hidden, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, hidden)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

# Full model

In [4]:
class BertAdapterClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # freeze BERT
        for p in self.bert.parameters():
            p.requires_grad = False

        self.adapter = Adapter()

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        x = out.last_hidden_state  # (B, T, 768)

        x = self.adapter(x)        # refine all tokens

        cls = x[:, 0]              # CLS token

        return self.classifier(cls)

# DataLoader

In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset["train"], batch_size=16, shuffle=True)
test_loader = DataLoader(dataset["test"], batch_size=16)

# Training setup

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertAdapterClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    list(model.adapter.parameters()) +
    list(model.classifier.parameters()),
    lr=2e-4
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6765.45it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Training loop

In [7]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)

        loss = criterion(logits, labels)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 291.8688
Epoch 2, Loss: 256.7435
Epoch 3, Loss: 248.1872


# Evaluation

In [8]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Accuracy:", correct / total)

Accuracy: 0.799


# Inference 

In [9]:
import torch

def predict_sentiment(text, model, tokenizer, device):
    model.eval()

    # tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=256
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # no gradient needed
    with torch.no_grad():
        logits = model(input_ids, attention_mask)

        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "positive" if pred == 1 else "negative"

    return label, probs.squeeze().cpu().numpy()

In [10]:
text = "This movie was absolutely amazing and emotional."

label, probs = predict_sentiment(text, model, tokenizer, device)

print("Prediction:", label)
print("Probabilities:", probs)

Prediction: positive
Probabilities: [0.01076283 0.98923725]
